# Front-End Pretraining — Neutral Head  (HPC full-training twin)

Full-resolution (`256^3`) pretraining of the **shared bi-planar front-end** (ConvNeXtV2 encoder +
hybrid fusion + 2D→3D lift) on the Sunway HPC GPU. The SimCLR-pretrained encoder stays **frozen**;
only the **fusion + lift** are trained with a throwaway **neutral SingleConv head** on the 3D
bone-occupancy task, then saved to `models/front_end.pth`.

`decoder_pipeline.ipynb` loads that file, **freezes the whole front-end**, and trains each decoder
on top — so U-Net and V-Net consume *byte-identical* features and the only difference is the decoder
block. This is the twin of `notebooks/modeling/frontend_pretrain.ipynb`; the **only** differences are
in the CONFIG cell.

Pipeline: `AP+LAT DRRs -> frozen SimCLR encoder -> fusion+lift (TRAIN) -> neutral SingleConv head -> occupancy volume`

### How to run

1. Run the cells top to bottom. The **CONFIG** cell is the only place you change settings.
2. This trains the front-end **once per fold** with the neutral head and writes
   `models/front_end_fold{FOLD}.pth` plus the per-fold split `models/decoders/decoder_split_fold{FOLD}.csv`.
3. For the cross-validated comparison, run this notebook for **each `FOLD` in `0..N_FOLDS-1`**, then
   run `decoder_pipeline.ipynb` (same `FOLD`, `REGIME="frozen"`) for `MODEL="unet"` and
   `MODEL="vnet"` — both load this fold's frozen front-end, so only the decoder differs.

This notebook requires the encoder checkpoint `models/convnextv2_simclr_encoder.pth` produced by
`encoder_pipeline.ipynb`. The front-end is pretrained **per fold** (it uses occupancy labels), so the
U-Net/V-Net comparison stays free of label leakage.

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # reduce CUDA fragmentation (OOM safeguard); must be set before torch initialises CUDA
import math, random, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as cp
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm
import nibabel as nib
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("torch", torch.__version__, "| timm", timm.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
# ============================= CONFIG (HPC - full 256^3 front-end pretraining) =============================
# Mirrors the local notebook; only these knobs differ. Run on the Sunway HPC GPU.
ENV          = "HPC"
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL        = "neutral"     # neutral SingleConv head: pretrains the shared front-end, favouring no decoder
TARGET_RES   = 256
LIFT_DEPTH   = 16            # MUST match decoder_pipeline.ipynb
EPOCHS       = 40
BATCH_SIZE   = 1             # 256^3 is memory-heavy; raise to 2 only if the GPU allows
LR           = 1e-4
CKPT_EVERY   = 5
USE_AMP      = True
USE_GRAD_CKPT = True
NUM_WORKERS  = 4
GT_THRESH    = 0.40
INCLUDE_GEOMETRIC = False    # MUST match decoder_pipeline.ipynb (same train split, no val/test leakage)
DEEP_SUPERVISION  = False    # not used by the neutral head
FREEZE_ENCODER = True        # freeze SimCLR backbone; train only the fusion + 2D->3D lift (+ neutral head)
# --- cross-validation (knee-level, dataset-stratified) — MUST match decoder_pipeline.ipynb ---
# The front-end uses occupancy GT (labels), so it is pretrained PER FOLD on that fold's train knees
# to keep the U-Net/V-Net comparison free of label leakage. This notebook is the PRODUCER of the
# per-fold split CSV that decoder_pipeline.ipynb then loads.
N_FOLDS      = 5             # k-fold CV over knees (matches FracReconNet); run every fold
FOLD         = 0             # which fold this front-end is for (0..N_FOLDS-1)
PRETRAINED   = True
SMOKE_TEST   = False
SMOKE_CASES_PER_GROUP = 3
RESUME_FROM  = None
EXPLICIT_ROOT = None         # e.g. "/home/project/xray2mesh/Marcus_Chan_Zheng_Shao_CP2 _24020059"
if DEVICE.type != "cuda":
    print("[warning] CUDA not available - this HPC notebook expects a GPU.")
print("ENV", ENV, "| MODEL", MODEL, "| fold", FOLD, "/", N_FOLDS,
      "| TARGET_RES", TARGET_RES, "| device", DEVICE, "| epochs", EPOCHS)

In [ ]:
# Resolve the project root robustly (works locally and on HPC, regardless of where
# the notebook is launched from). We look upward for the data/interim/predrr folder,
# which holds the ground-truth CT volumes.
def find_root(start: Path) -> Path:
    if EXPLICIT_ROOT:
        r = Path(EXPLICIT_ROOT)
        if (r / "data" / "interim" / "predrr").exists():
            return r
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / "data" / "interim" / "predrr").exists():
            return cand
    raise FileNotFoundError("Could not find project root (expected data/interim/predrr). "
                            "Set EXPLICIT_ROOT in the CONFIG cell.")

ROOT           = find_root(Path.cwd())
DATA           = ROOT / "data"
NORMAL_DRR_DIR = DATA / "interim" / "DRRs"                 # AP/LAT DRRs (model inputs)
AUG_DRR_DIR    = DATA / "processed" / "augmented_DRRs"     # augmented DRR variants
PREDRR_DIR     = DATA / "interim" / "predrr"               # ground-truth CT volumes
MODELS_DIR     = ROOT / "models"
SIMCLR_CKPT    = MODELS_DIR / "convnextv2_simclr_encoder.pth"      # encoder weights (fold-agnostic; SSL uses no labels)
FRONTEND_CKPT  = MODELS_DIR / ("front_end_fold%d.pth" % FOLD)      # <- per-fold output; decoder_pipeline.ipynb loads the matching FOLD
CKPT_DIR       = MODELS_DIR / "decoders" / ("neutral_fold%d" % FOLD)   # per-fold neutral-head checkpoints (intermediate)
CKPT_DIR.mkdir(parents=True, exist_ok=True)
GT_CACHE_DIR   = DATA / "interim" / ("predrr_occupancy_%d" % TARGET_RES)   # cached binary GT
print("ROOT:", ROOT)
print("front-end ->", FRONTEND_CKPT)
print("checkpoints ->", CKPT_DIR)

## 1. Shared encoder (copied verbatim from `encoder_pipeline.ipynb`)

The encoder is the part both decoders share, so the comparison is fair: **only the decoder
changes**. The code below is copied verbatim from `encoder_pipeline.ipynb` so this notebook is
self-contained — **do not edit it here**. (When `encoder_pipeline.ipynb` is later converted to a
`.py` module, replace this cell with a simple `import`.)

What it does, in plain terms:
1. A **ConvNeXtV2** backbone turns each X-ray (AP and LAT) into 4 feature maps at increasing depth.
2. **Hybrid bi-planar fusion** merges the two views: cheap convolution at fine scales (keeps local
   fracture detail), cross-attention at coarse scales (aligns global knee shape).
3. A **2D->3D lift** stacks each fused map into a small 3D feature volume (depth = `LIFT_DEPTH`).

Output: a list of 4 multi-scale 3D feature tensors with channels `[64, 128, 256, 512]` — this is
the *contract* the decoder consumes.

In [ ]:
# ===== Encoder front-end - VERBATIM from encoder_pipeline.ipynb. DO NOT EDIT. =====
# (PRETRAINED / FREEZE_ENCODER are set in the CONFIG cell so they stay visible knobs.)
BACKBONE     = "convnextv2_tiny"
IMG_SIZE     = 256
OUT_CHANNELS = [64, 128, 256, 512]
FUSION_TYPES = ["local", "local", "attn", "attn"]   # fine -> coarse

def make_backbone(pretrained=True):
    """features_only ConvNeXtV2 returning 4 multi-scale maps. Falls back to random init offline."""
    try:
        return timm.create_model(BACKBONE, pretrained=pretrained, features_only=True)
    except Exception as e:
        print("[warn] pretrained fetch failed (%s); random init." % type(e).__name__)
        return timm.create_model(BACKBONE, pretrained=False, features_only=True)

FEAT_DIMS = [f["num_chs"] for f in make_backbone(pretrained=False).feature_info]   # [96,192,384,768]

def load_drr(path):
    """npy 256x256 float32 [0,1] -> tensor [3,H,W] (1 channel replicated to 3 for ConvNeXtV2)."""
    arr = np.load(path).astype(np.float32)
    t = torch.from_numpy(arr)
    if t.ndim == 2:
        t = t.unsqueeze(0)
    return t.repeat(3, 1, 1) if t.shape[0] == 1 else t

NORMALIZE = T.Normalize(mean=[0.5] * 3, std=[0.5] * 3)
def paired_tf(t):
    return NORMALIZE(t)

class CrossAttention(nn.Module):
    """AP (query) attends to LAT (key/value). Operates on tokens [B, N, C]."""
    def __init__(self, dim):
        super().__init__()
        self.q = nn.Linear(dim, dim); self.k = nn.Linear(dim, dim); self.v = nn.Linear(dim, dim)
        self.scale = dim ** -0.5
    def forward(self, a, b):
        attn = F.softmax(torch.matmul(self.q(a), self.k(b).transpose(-2, -1)) * self.scale, dim=-1)
        return torch.matmul(attn, self.v(b)) + a

class LocalFusion(nn.Module):
    """Cheap high-res fusion: concat views + 3x3 conv, residual on AP."""
    def __init__(self, dim):
        super().__init__()
        self.mix = nn.Conv2d(2 * dim, dim, kernel_size=3, padding=1)
    def forward(self, a, b):
        return self.mix(torch.cat([a, b], dim=1)) + a

# --- bi-planar lift orientation (resolved empirically; see Check 1 / _axis_probe) ---
# GT array axes, from nibabel axcodes ('L','P','S'): axis0 = L-R, axis1 = A-P, axis2 = S-I.
# AP projects along A-P (axis1); LAT along L-R (axis0). Both DRR rows (H) = S-I (axis2);
# AP cols (W) = L-R (axis0); LAT cols (W) = A-P (axis1). Cube ordered (axis0,axis1,axis2) to
# match the GT array. flip_* reverse a row/col vs its volume axis; confirmed by overfit guard.
LIFT_FLIP_SI      = False   # DRR rows  vs axis2 (S-I)
LIFT_FLIP_AP_COL  = False   # AP  cols  vs axis0 (L-R)
LIFT_FLIP_LAT_COL = True    # LAT cols  vs axis1 (A-P)

class BiPlanarFeatureFusion(nn.Module):
    def __init__(self, feat_dims=FEAT_DIMS, out_channels=OUT_CHANNELS,
                 fusion_types=FUSION_TYPES, depth=16, pretrained=True, freeze_encoder=False):
        super().__init__()
        self.encoder = make_backbone(pretrained)
        self.fusion_types = list(fusion_types); self.depth = depth
        self.fuse = nn.ModuleList([CrossAttention(d) if t == "attn" else LocalFusion(d)
                                   for d, t in zip(feat_dims, fusion_types)])
        self.to3d = nn.ModuleList([nn.Conv2d(c, o, 1) for c, o in zip(feat_dims, out_channels)])
        self.expand3d = nn.ModuleList([nn.Conv3d(2 * o, o, 3, padding=1) for o in out_channels])
        if freeze_encoder:
            for p in self.encoder.parameters():
                p.requires_grad = False
    def load_simclr_encoder(self, path):
        missing, unexpected = self.encoder.load_state_dict(torch.load(path, map_location="cpu"), strict=False)
        print("loaded SimCLR encoder: missing=%d unexpected=%d" % (len(missing), len(unexpected)))
    def _ortho_lift(self, ap_f, lat_f, c2d, c3d):
        """Orthogonal back-projection lift: place each view on the two volume axes it resolves
        and broadcast along its unobserved projection axis, then fuse the two cubes in 3D.
        Output cube ordered (axis0=L-R, axis1=A-P, axis2=S-I) to match the GT array."""
        B, C, H, W = ap_f.shape           # square feature map at this level: S = H = W
        S = H
        ap = c2d(ap_f); lat = c2d(lat_f)  # shared 1x1 projection -> [B, O, H, W] each
        O = ap.shape[1]
        if LIFT_FLIP_SI:      ap = ap.flip(2); lat = lat.flip(2)   # rows (H) = S-I (axis2)
        if LIFT_FLIP_AP_COL:  ap = ap.flip(3)                       # AP  cols (W) = L-R (axis0)
        if LIFT_FLIP_LAT_COL: lat = lat.flip(3)                     # LAT cols (W) = A-P (axis1)
        ap_cube = ap.permute(0, 1, 3, 2).unsqueeze(3).expand(B, O, S, S, S)   # broadcast axis1 (A-P)
        lat_cube = lat.permute(0, 1, 3, 2).unsqueeze(2).expand(B, O, S, S, S)  # broadcast axis0 (L-R)
        return c3d(torch.cat([ap_cube, lat_cube], dim=1))           # [B, O, S, S, S]
    def forward(self, ap_img, lat_img):
        ap_feats, lat_feats = self.encoder(ap_img), self.encoder(lat_img)
        fused2d, fused3d = [], []
        for ap_f, lat_f, fuse, c2d, c3d, t in zip(
                ap_feats, lat_feats, self.fuse, self.to3d, self.expand3d, self.fusion_types):
            B, C, H, W = ap_f.shape
            if t == "attn":
                a = ap_f.flatten(2).transpose(1, 2); b = lat_f.flatten(2).transpose(1, 2)
                f2d = fuse(a, b).transpose(1, 2).reshape(B, C, H, W)
            else:
                f2d = fuse(ap_f, lat_f)
            fused2d.append(f2d)
            fused3d.append(self._ortho_lift(ap_f, lat_f, c2d, c3d))
        return fused2d, fused3d

print("encoder feature dims:", FEAT_DIMS)

## 2. Data — paired (DRR inputs, binary-occupancy GT)

The (X-ray, CT) pairs are aligned *by construction*: the DRRs were rendered **from** these exact CT
volumes. So for each DRR pair we look up the matching `predrr` CT volume and turn it into the
training target.

**Ground-truth target = binary bone occupancy.** The `predrr` CT is bone-windowed and scaled to
`[0,1]`. We threshold it at `GT_THRESH` to get a `{0,1}` bone mask, resample it to `TARGET_RES^3`
with **nearest-neighbour** (so it stays binary), and cache it to disk (so we threshold once, not
every epoch). Tune `GT_THRESH` using the QA cell near the bottom.

**Splitting** is done at the *knee* level (`dataset, case, side`) so all augmented variants of one
knee land in the same split (no leakage), and healthy/fractured cases are stratified across
train/val/test. We exclude **geometric** augmentations (rotations/flips) by default, because their
3D ground truth would need the same transform applied — photometric variants reuse the base GT.

In [ ]:
def build_paired_index():
    """One row per (case, side, variant) with absolute AP/LAT paths + metadata."""
    rows = []
    nmeta = pd.read_csv(NORMAL_DRR_DIR / "drr_generation_metadata.csv")
    for (ds, case, side), _ in nmeta.groupby(["dataset", "case", "side"]):
        ap = NORMAL_DRR_DIR / ds / case / side / "ap.npy"
        lat = NORMAL_DRR_DIR / ds / case / side / "lat.npy"
        if ap.exists() and lat.exists():
            rows.append(dict(dataset=ds, case=case, side=side, variant="normal",
                             geometric=False, ap=str(ap), lat=str(lat)))
    ameta_path = AUG_DRR_DIR / "augmentation_variants_metadata.csv"
    if ameta_path.exists():
        ameta = pd.read_csv(ameta_path)
        for r in ameta.itertuples(index=False):
            ap = AUG_DRR_DIR / r.ap_npy; lat = AUG_DRR_DIR / r.lat_npy
            if ap.exists() and lat.exists():
                rows.append(dict(dataset=r.dataset, case=r.case, side=r.side, variant=r.variant,
                                 geometric=bool(r.geometric), ap=str(ap), lat=str(lat)))
    return pd.DataFrame(rows)

paired_index = build_paired_index()
if not INCLUDE_GEOMETRIC:
    paired_index = paired_index[~paired_index.geometric].reset_index(drop=True)
print("paired rows:", len(paired_index),
      "| cases:", paired_index.groupby("dataset")["case"].nunique().to_dict())

# ---- ground truth: predrr CT -> binary bone occupancy at TARGET_RES (cached) ----
def gt_file(dataset, case, side):
    Side = "Right" if str(side).lower().startswith("r") else "Left"
    if dataset == "healthy":
        return PREDRR_DIR / "healthy" / ("%s_%s.nii.gz" % (case, Side))
    return PREDRR_DIR / "fractured" / ("%s_Part%s.nii.gz" % (case, Side))

def load_gt_occupancy(dataset, case, side):
    GT_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    cache = GT_CACHE_DIR / ("%s_%s_%s.npy" % (dataset, case, side))
    if cache.exists():
        occ = np.load(cache)
    else:
        vol = nib.load(str(gt_file(dataset, case, side))).get_fdata().astype(np.float32)  # 256^3 [0,1]
        occ = (vol > GT_THRESH).astype(np.float32)
        t = F.interpolate(torch.from_numpy(occ)[None, None], size=(TARGET_RES,) * 3, mode="nearest")
        occ = t[0, 0].numpy().astype(np.float32)
        np.save(cache, occ)
    return torch.from_numpy(occ)[None]   # (1, T, T, T)

class PairedDRRVolumeDataset(Dataset):
    """Returns AP/LAT DRRs (3x256x256) + binary GT occupancy (1,T,T,T) + metadata."""
    def __init__(self, df, transform=paired_tf):
        self.df = df.reset_index(drop=True); self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        ap = self.transform(load_drr(r.ap)); lat = self.transform(load_drr(r.lat))
        gt = load_gt_occupancy(r.dataset, r.case, r.side)
        return {"ap": ap, "lat": lat, "gt": gt,
                "dataset": r.dataset, "case": r.case, "side": r.side, "variant": r.variant}

# ---- knee-level K-FOLD cross-validation split (dataset-stratified) ----
# This notebook PRODUCES the per-fold split CSV the decoder loads. 5-fold CV (as in FracReconNet,
# PMC9829664) over (case, side) puts every knee in a test fold exactly once, so all 13 fractured
# knees get tested across the run -- a single 70/15/15 split would leave only ~2 fractured test
# knees, far too few for a credible conclusion (RSNA Radiology:AI 2022 small-sample guidance).
# Round-robin slicing keeps fold sizes balanced; test = fold k, val = fold (k+1), train = the rest.
def kfold_split(df, n_folds=N_FOLDS, fold=FOLD, seed=SEED):
    rng = random.Random(seed); assign = {}
    for ds, g in df.groupby("dataset"):
        keys = sorted({(r.case, r.side) for r in g.itertuples()})
        rng.shuffle(keys)
        folds = [keys[i::n_folds] for i in range(n_folds)]      # round-robin -> balanced sizes
        test_keys = set(folds[fold % n_folds]); val_keys = set(folds[(fold + 1) % n_folds])
        for k in keys:
            kk = (ds,) + k
            assign[kk] = "test" if k in test_keys else ("val" if k in val_keys else "train")
    return df.apply(lambda r: assign[(r.dataset, r.case, r.side)], axis=1)

if SMOKE_TEST:
    # tiny, fast subset just to verify the pipeline runs end-to-end (keep FOLD=0 in smoke)
    keep = paired_index[paired_index.variant == "normal"]
    sub = [g[g.case.isin(list(dict.fromkeys(g.case))[:SMOKE_CASES_PER_GROUP])]
           for _, g in keep.groupby("dataset")]
    paired_index = pd.concat(sub).reset_index(drop=True)

paired_index["split"] = kfold_split(paired_index)
split_csv = MODELS_DIR / "decoders" / ("decoder_split_fold%d.csv" % FOLD)
split_csv.parent.mkdir(parents=True, exist_ok=True)
paired_index[["dataset", "case", "side", "variant", "split"]].to_csv(split_csv, index=False)
print("wrote fold %d split -> %s" % (FOLD, split_csv.name))
print(paired_index.groupby(["split", "dataset"]).size())

In [ ]:
train_df = paired_index[paired_index.split == "train"]
# Evaluate on the CLEAN DRR only: photometric augmented variants are a TRAIN-time augmentation.
# Keeping them in val/test would duplicate each held-out knee as several near-identical rows,
# making the per-knee metric (and the U-Net vs V-Net pairing downstream) ambiguous.
val_df   = paired_index[(paired_index.split == "val")  & (paired_index.variant == "normal")]
test_df  = paired_index[(paired_index.split == "test") & (paired_index.variant == "normal")]

def make_loader(df, shuffle):
    if len(df) == 0:
        return None
    return DataLoader(PairedDRRVolumeDataset(df), batch_size=BATCH_SIZE,
                      shuffle=shuffle, num_workers=NUM_WORKERS, drop_last=False)

train_loader = make_loader(train_df, True)
val_loader   = make_loader(val_df, False)
test_loader  = make_loader(test_df, False)
print("samples -> train:", len(train_df), "| val:", len(val_df), "| test:", len(test_df),
      "(val/test = normal variant only)")

## 3. Neutral pretraining head

The head reuses the **exact decoder wiring** both contenders share — upsample step by step
(`8 -> 16 -> 32 -> 64`) with encoder features concatenated as **skip connections**, then a
**super-resolution head** grows the `(LIFT_DEPTH, 64, 64)` grid up to `TARGET_RES^3`. The **only**
difference from the real decoders is the building block:

- **`neutral`** -> `SingleConv`: one `Conv3d -> BatchNorm -> ReLU`.
- (`unet` -> `DoubleConv` = two of these; `vnet` -> `VNetResBlock` = two + residual — defined here
  too, but **not used** in this notebook.)

Training with `SingleConv` gives the frozen encoder's fusion + 2D→3D lift a gradient signal toward
the occupancy task without tuning them to the quirks of either decoder. After training we save the
front-end and **discard this head**.

In [ ]:
def conv_block(block_type, in_ch, out_ch):
    if block_type == "unet":
        return DoubleConv(in_ch, out_ch)
    if block_type == "vnet":
        return VNetResBlock(in_ch, out_ch)
    if block_type == "neutral":
        return SingleConv(in_ch, out_ch)
    raise ValueError("unknown block_type: %s" % block_type)

class SingleConv(nn.Module):
    """Neutral block: a single Conv3d -> BN -> ReLU. The common ancestor of U-Net's DoubleConv
    (two of these) and V-Net's residual block (two + a skip), so pretraining the front-end with it
    favours neither decoder. Used ONLY to give the fusion + 2D->3D lift a gradient signal; the head
    itself is discarded after pretraining."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1), nn.BatchNorm3d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x):
        return self.net(x)

class DoubleConv(nn.Module):
    """U-Net block: (Conv3d -> BN -> ReLU) x2. Plain, no residual."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1), nn.BatchNorm3d(out_ch), nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1), nn.BatchNorm3d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x):
        return self.net(x)

class VNetResBlock(nn.Module):
    """V-Net block: (Conv3d -> BN -> PReLU) x2 + residual add (input projected if channels differ)."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.proj = nn.Conv3d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.c1 = nn.Conv3d(in_ch, out_ch, 3, padding=1); self.n1 = nn.BatchNorm3d(out_ch); self.a1 = nn.PReLU(out_ch)
        self.c2 = nn.Conv3d(out_ch, out_ch, 3, padding=1); self.n2 = nn.BatchNorm3d(out_ch); self.a2 = nn.PReLU(out_ch)
    def forward(self, x):
        y = self.a1(self.n1(self.c1(x)))
        y = self.n2(self.c2(y))
        return self.a2(y + self.proj(x))

class SuperResHead(nn.Module):
    """Grow the (LIFT_DEPTH, 64, 64) feature grid up to (T,T,T) via staged trilinear upsample +
    refine blocks with tapering channels (heavy work stays at low resolution -> low memory)."""
    def __init__(self, block_type, in_ch, target, use_grad_ckpt=False):
        super().__init__()
        self.target = tuple(int(t) for t in target); self.use_grad_ckpt = use_grad_ckpt
        self.b1 = conv_block(block_type, in_ch, 32)
        self.b2 = conv_block(block_type, 32, 16)
        self.b3 = conv_block(block_type, 16, 8)
        self.out = nn.Conv3d(8, 1, 1)
    def _run(self, blk, x):
        if self.use_grad_ckpt and x.requires_grad:
            return cp.checkpoint(blk, x, use_reentrant=False)
        return blk(x)
    def forward(self, x):
        d0, h0, w0 = x.shape[-3:]; dt, ht, wt = self.target
        s1 = (round(d0 + (dt - d0) / 3), round(h0 + (ht - h0) / 3), round(w0 + (wt - w0) / 3))
        s2 = (round(d0 + 2 * (dt - d0) / 3), round(h0 + 2 * (ht - h0) / 3), round(w0 + 2 * (wt - w0) / 3))
        x = F.interpolate(x, size=s1, mode="trilinear", align_corners=False); x = self._run(self.b1, x)
        x = F.interpolate(x, size=s2, mode="trilinear", align_corners=False); x = self._run(self.b2, x)
        x = F.interpolate(x, size=self.target, mode="trilinear", align_corners=False); x = self._run(self.b3, x)
        return self.out(x)

class Decoder3D(nn.Module):
    """Multi-scale skip-connected decoder. Same wiring for every block type; only the block differs
    (unet=DoubleConv, vnet=VNetResBlock, neutral=SingleConv)."""
    def __init__(self, block_type, enc_channels=OUT_CHANNELS, target=(64, 64, 64),
                 use_grad_ckpt=False, deep_supervision=False):
        super().__init__()
        c0, c1, c2, c3 = enc_channels
        self.deep_supervision = deep_supervision; self.target = tuple(int(t) for t in target)
        self.up3 = nn.ConvTranspose3d(c3, c2, kernel_size=2, stride=2)
        self.dec3 = conv_block(block_type, c2 + c2, c2)
        self.up2 = nn.ConvTranspose3d(c2, c1, kernel_size=2, stride=2)
        self.dec2 = conv_block(block_type, c1 + c1, c1)
        self.up1 = nn.ConvTranspose3d(c1, c0, kernel_size=2, stride=2)
        self.dec1 = conv_block(block_type, c0 + c0, c0)
        self.sr = SuperResHead(block_type, c0, self.target, use_grad_ckpt)
        if deep_supervision:
            self.aux3 = nn.Conv3d(c2, 1, 1); self.aux2 = nn.Conv3d(c1, 1, 1); self.aux1 = nn.Conv3d(c0, 1, 1)
    def forward(self, feats):
        l0, l1, l2, l3 = feats
        x = self.up3(l3); x = torch.cat([x, l2], 1); x = self.dec3(x); a3 = x
        x = self.up2(x);  x = torch.cat([x, l1], 1); x = self.dec2(x); a2 = x
        x = self.up1(x);  x = torch.cat([x, l0], 1); x = self.dec1(x); a1 = x
        out = self.sr(x)
        if self.deep_supervision and self.training:
            up = lambda h: F.interpolate(h, size=self.target, mode="trilinear", align_corners=False)
            return out, [up(self.aux3(a3)), up(self.aux2(a2)), up(self.aux1(a1))]
        return out, None

class ReconModel(nn.Module):
    """Full model = shared bi-planar encoder/fusion + a decoder (here the neutral pretraining head)."""
    def __init__(self, fusion, decoder):
        super().__init__(); self.fusion = fusion; self.decoder = decoder
    def forward(self, ap, lat):
        _, f3d = self.fusion(ap, lat)
        return self.decoder(f3d)

In [ ]:
def build_model():
    fusion = BiPlanarFeatureFusion(depth=LIFT_DEPTH, pretrained=PRETRAINED, freeze_encoder=FREEZE_ENCODER)
    if SIMCLR_CKPT.exists():
        fusion.load_simclr_encoder(SIMCLR_CKPT)
    else:
        print("[warn] no SimCLR encoder checkpoint; encoder uses ImageNet/random init.")
    decoder = Decoder3D(MODEL, target=(TARGET_RES,) * 3,
                        use_grad_ckpt=USE_GRAD_CKPT, deep_supervision=DEEP_SUPERVISION)
    return ReconModel(fusion, decoder)

# shape sanity check (eval mode, no grad -> cheap)
_m = build_model().to(DEVICE).eval()
with torch.no_grad():
    _ap = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    _out, _ = _m(_ap, _ap)
print("model:", MODEL, "| output volume:", tuple(_out.shape),
      "| expected:", (1, 1, TARGET_RES, TARGET_RES, TARGET_RES))
del _m, _ap, _out

## Diagnostics — smoking-gun checks (assert + visualize)

Two guards that would have caught the extrusion bug (3D features constant along depth → Dice stuck
at ~0.45) and that fail loudly on a regression:

- **Check 1 — GT↔DRR axis alignment.** Back-projects both DRRs through the lift's own geometry and
  asserts their intersection actually contains the bone (recall > 0.30). Also shows the two DRRs
  next to the GT projections so you can eyeball that AP↔axis1 and LAT↔axis0.
- **Check 2 — per-axis feature variance.** Asserts the 3D features vary along **all three** spatial
  axes (std > 1e-4). The old extruding lift produced std≈0 along depth — this is the direct detector.

In [ ]:
# ===== Check 1 - GT<->DRR axis alignment (visual + hull-recall guard) =====
# Back-project both DRRs exactly as _ortho_lift does (O=1, no conv) and intersect them. With the
# configured orientation the bone must fall INSIDE the hull -> high recall. A gross axis/flip error
# separates the two shadows and recall collapses. (Subtle flips were settled by a one-sample overfit
# test; this cell guards against gross mis-orientation and lets you eyeball DRR-vs-GT alignment.)
@torch.no_grad()
def _hull_recall(ap2d, lat2d, occ, res=96):
    a = torch.from_numpy(ap2d)[None, None].float(); l = torch.from_numpy(lat2d)[None, None].float()
    a = F.interpolate(a, (res, res), mode="bilinear", align_corners=False)
    l = F.interpolate(l, (res, res), mode="bilinear", align_corners=False)
    a = (a - a.min()) / (a.max() - a.min() + 1e-8); l = (l - l.min()) / (l.max() - l.min() + 1e-8)
    if LIFT_FLIP_SI:      a = a.flip(2); l = l.flip(2)
    if LIFT_FLIP_AP_COL:  a = a.flip(3)
    if LIFT_FLIP_LAT_COL: l = l.flip(3)
    S = res
    ac = a.permute(0, 1, 3, 2).unsqueeze(3).expand(1, 1, S, S, S)   # AP  -> broadcast axis1 (A-P)
    lc = l.permute(0, 1, 3, 2).unsqueeze(2).expand(1, 1, S, S, S)   # LAT -> broadcast axis0 (L-R)
    hull = (ac * lc)[0, 0].numpy()
    occ_r = F.interpolate(torch.from_numpy(occ.astype(np.float32))[None, None], (S, S, S),
                          mode="nearest")[0, 0].numpy() > 0.5
    k = max(int(3 * occ_r.sum()), 1)                                # keep top ~3x bone-count voxels
    thr = np.partition(hull.ravel(), -k)[-k]
    return ((hull >= thr) & occ_r).sum() / max(occ_r.sum(), 1)

_s = paired_index.iloc[0]
_occ = nib.load(str(gt_file(_s.dataset, _s.case, _s.side))).get_fdata().astype(np.float32) > GT_THRESH
_ap_img = np.load(_s.ap).astype(np.float32); _lat_img = np.load(_s.lat).astype(np.float32)
_rec = _hull_recall(_ap_img, _lat_img, _occ)
print("Check 1 - back-projected hull recall of GT = %.3f  (orientation SI=%s AP=%s LAT=%s; case %s %s)"
      % (_rec, LIFT_FLIP_SI, LIFT_FLIP_AP_COL, LIFT_FLIP_LAT_COL, _s.case, _s.side))
fig, ax = plt.subplots(1, 5, figsize=(16, 3.2))
ax[0].imshow(_ap_img, cmap="gray"); ax[0].set_title("AP DRR")
ax[1].imshow(_lat_img, cmap="gray"); ax[1].set_title("LAT DRR")
for j, k in enumerate((0, 1, 2)):
    ax[2 + j].imshow(_occ.max(axis=k), cmap="gray"); ax[2 + j].set_title("GT MIP axis%d" % k)
for a in ax:
    a.axis("off")
plt.suptitle("Check 1 - inputs vs GT projections (lift expects AP->axis1, LAT->axis0)")
plt.tight_layout(); plt.show()
assert _rec > 0.30, "Check 1 FAIL: configured back-projection misses the bone (recall<0.30) - orientation likely wrong"
print("Check 1 PASS: the configured orthogonal back-projection contains the bone.\n")

In [ ]:
# ===== Check 2 - per-axis feature variance (the extrusion detector) =====
# The reference lift broadcast one fused 2D map along depth -> std==0 along that axis (the bug that
# capped Dice ~0.45). The orthogonal lift must vary along ALL THREE spatial axes. FAIL loudly if any
# axis collapses, so a regression to an extruding lift cannot pass a run silently.
_diag = build_model().to(DEVICE).eval()
_rows = []
with torch.no_grad():
    for ds in ["healthy", "fractured"]:
        sub = paired_index[paired_index.dataset == ds]
        if len(sub) == 0:
            continue
        r = sub.iloc[0]
        ap = paired_tf(load_drr(r.ap)).unsqueeze(0).to(DEVICE)
        lat = paired_tf(load_drr(r.lat)).unsqueeze(0).to(DEVICE)
        _, f3d = _diag.fusion(ap, lat)
        for i, f in enumerate(f3d):
            s0, s1, s2 = f.std(2).mean().item(), f.std(3).mean().item(), f.std(4).mean().item()
            _rows.append((ds, i, s0, s1, s2, min(s0, s1, s2) > 1e-4))
print("Check 2 - per-axis feature std (must be > 1e-4 on every axis):")
for ds, i, s0, s1, s2, ok in _rows:
    print("  %-9s level%d: std[axis0]=%.4f std[axis1]=%.4f std[axis2]=%.4f  %s"
          % (ds, i, s0, s1, s2, "OK" if ok else "FAIL - axis collapsed (extrusion!)"))
assert _rows and all(r[5] for r in _rows), "Check 2 FAIL: a 3D feature is constant along an axis (extrusion bug)"
print("Check 2 PASS: 3D features vary on all three axes (no extrusion).\n")
del _diag

In [ ]:
# --- NaN diagnostic (optional): finds the first non-finite tensor. Set False to skip. ---
# If training ever prints NaN, run this: it shows whether the inputs, the fused 3D features, or the
# logits become non-finite, and whether float16 autocast (vs fp32) is what introduces it.
RUN_NAN_DIAGNOSTIC = True
if RUN_NAN_DIAGNOSTIC and train_loader is not None:
    _dm = build_model().to(DEVICE).eval()
    _b = next(iter(train_loader))
    _ap, _lat, _gt = _b["ap"].to(DEVICE), _b["lat"].to(DEVICE), _b["gt"].to(DEVICE)
    def _chk(name, t):
        print("  %-12s finite=%s min=%.3g max=%.3g"
              % (name, bool(torch.isfinite(t).all()), float(t.min()), float(t.max())))
    print("inputs:"); _chk("ap", _ap); _chk("lat", _lat); _chk("gt", _gt)
    for use in ([False, True] if DEVICE.type == "cuda" else [False]):
        ad = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        with torch.no_grad():
            if use:
                with torch.amp.autocast("cuda", dtype=ad):
                    _f2, _f3 = _dm.fusion(_ap, _lat); _o, _ = _dm.decoder(_f3)
            else:
                _f2, _f3 = _dm.fusion(_ap, _lat); _o, _ = _dm.decoder(_f3)
        print("autocast", use, "| dtype", (str(ad) if use else "fp32"))
        for i, f in enumerate(_f3):
            _chk("feat3d[%d]" % i, f)
        _chk("logits", _o)
    del _dm, _b, _ap, _lat, _gt

## 4. Loss and metrics

- **Loss = 0.5 * BCE + 0.5 * soft-Dice.** Bone is a small fraction of the volume (~3%), so pure BCE
  is dominated by easy background voxels. Adding Dice directly optimises overlap and handles the
  class imbalance. (Lai's reference used Dice only; adding BCE stabilises early training.)
- **Metrics:** **Dice** and **IoU** (overlap), and **HD95** (95th-percentile surface distance, in
  voxels) for boundary accuracy. We compute them on the *binarised* prediction. We report them
  **overall and split by healthy vs fractured**, which is what the research goal needs.

We deliberately implement these by hand (instead of pulling in MONAI) so the formulas are visible
and the notebook runs with the libraries already installed.

In [ ]:
class DiceBCELoss(nn.Module):
    def __init__(self, w_bce=0.5, w_dice=0.5, smooth=1.0):
        super().__init__(); self.w_bce = w_bce; self.w_dice = w_dice; self.smooth = smooth
    def _dice(self, logits, target):
        # IMPORTANT: reduce in float32. Under AMP the logits are float16, and at 256^3 a
        # float16 sum of ~16.8M voxels overflows (>65504) -> inf -> Dice becomes NaN.
        p = torch.sigmoid(logits.float()).reshape(logits.size(0), -1)
        t = target.float().reshape(target.size(0), -1)
        inter = (p * t).sum(1); union = p.sum(1) + t.sum(1)
        return 1 - ((2 * inter + self.smooth) / (union + self.smooth)).mean()
    def forward(self, logits, target):
        logits = logits.float()   # compute the loss in fp32 even when the forward pass ran in fp16
        return self.w_bce * F.binary_cross_entropy_with_logits(logits, target.float()) + self.w_dice * self._dice(logits, target)

DICE_BCE = DiceBCELoss()

def total_loss(output, target, aux_weight=0.3):
    out, aux = output
    loss = DICE_BCE(out, target)
    if aux:
        for a in aux:
            loss = loss + aux_weight * DICE_BCE(a, target)
    return loss

@torch.no_grad()
def dice_iou(logits, target, thr=0.5):
    p = (torch.sigmoid(logits.float()) > thr).float().reshape(logits.size(0), -1)
    t = (target > 0.5).float().reshape(target.size(0), -1)
    inter = (p * t).sum(1); psum = p.sum(1); tsum = t.sum(1)
    dice = (2 * inter + 1e-6) / (psum + tsum + 1e-6)
    iou = (inter + 1e-6) / (psum + tsum - inter + 1e-6)
    return dice.cpu().numpy(), iou.cpu().numpy()

def hd95(pred_bin, gt_bin):
    """95th-percentile symmetric surface distance in voxels. Inputs: 3D bool arrays."""
    from scipy.ndimage import binary_erosion, distance_transform_edt
    sp = pred_bin & ~binary_erosion(pred_bin); sg = gt_bin & ~binary_erosion(gt_bin)
    if sp.sum() == 0 or sg.sum() == 0:
        return float("nan")
    dg = distance_transform_edt(~sg); dp = distance_transform_edt(~sp)
    return float(np.percentile(np.concatenate([dg[sp], dp[sg]]), 95))

In [ ]:
def run_epoch(model, loader, optimizer, scaler, train):
    model.train(train)
    use_amp = USE_AMP and DEVICE.type == "cuda"
    # bfloat16 has float32's dynamic range -> no overflow at 65504 (the float16 NaN cause).
    amp_dtype = torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported()) else torch.float16
    use_scaler = use_amp and amp_dtype == torch.float16   # only float16 needs GradScaler
    tot, n, steps, skipped = 0.0, 0, 0, 0
    for batch in loader:
        ap = batch["ap"].to(DEVICE); lat = batch["lat"].to(DEVICE); gt = batch["gt"].to(DEVICE)
        with torch.set_grad_enabled(train):
            if use_amp:
                with torch.amp.autocast("cuda", dtype=amp_dtype):
                    loss = total_loss(model(ap, lat), gt)
            else:
                loss = total_loss(model(ap, lat), gt)
        if not torch.isfinite(loss):                       # never backprop a NaN/inf loss
            skipped += 1; optimizer.zero_grad(set_to_none=True); continue
        if train:
            optimizer.zero_grad(set_to_none=True)
            if use_scaler:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                prev = scaler.get_scale(); scaler.step(optimizer); scaler.update()
                if scaler.get_scale() >= prev: steps += 1   # scaler did not skip the step
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); steps += 1
        tot += loss.item() * ap.size(0); n += ap.size(0)
    if skipped: print("  [warn] skipped %d non-finite batch(es)" % skipped)
    return tot / max(n, 1), steps

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); rows = []
    for batch in loader:
        out, _ = model(batch["ap"].to(DEVICE), batch["lat"].to(DEVICE))
        d, i = dice_iou(out, batch["gt"].to(DEVICE))
        for b in range(len(d)):
            rows.append(dict(dataset=batch["dataset"][b], case=batch["case"][b],
                             side=batch["side"][b], dice=float(d[b]), iou=float(i[b])))
    df = pd.DataFrame(rows)
    overall = df[["dice", "iou"]].mean().to_dict() if len(df) else {"dice": float("nan"), "iou": float("nan")}
    by = df.groupby("dataset")[["dice", "iou"]].mean() if len(df) else None
    return df, overall, by

def save_ckpt(path, model, optimizer, scheduler, epoch, val_metrics):
    torch.save({"epoch": epoch, "model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict() if scheduler else None, "val_metrics": val_metrics,
                "config": {"MODEL": MODEL, "TARGET_RES": TARGET_RES, "LIFT_DEPTH": LIFT_DEPTH,
                           "GT_THRESH": GT_THRESH}}, path)

## 5. Training with checkpointing

Each epoch we train, validate, score Dice/IoU on the validation set, step the cosine LR schedule,
and **save checkpoints**:
- `MODEL_last.pth` — always the most recent (for resuming),
- `MODEL_epochNNN.pth` — every `CKPT_EVERY` epochs (so you can **revisit any epoch** later),
- `MODEL_best.pth` — whenever validation Dice improves,
- `MODEL_history.csv` — per-epoch losses/metrics for the learning-curve plot.

Each checkpoint stores the epoch, model + optimizer + scheduler state, the validation metrics, and
the run config (model type, resolution, lift depth, GT threshold) — everything needed to resume or
to load the model later in the UI. On GPU we use **AMP** (mixed precision) and optional **gradient
checkpointing** to fit `256^3` in memory.

In [ ]:
model = build_model().to(DEVICE)
# encoder is frozen (FREEZE_ENCODER); train only the fusion + 2D->3D lift + neutral head
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(EPOCHS, 1))
scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and DEVICE.type == "cuda"))

n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print("trainable params: %.2fM (fusion+lift+neutral head) | frozen: %.2fM (encoder)"
      % (n_train / 1e6, n_frozen / 1e6))

start_epoch, best_dice, history = 0, -1.0, []
if RESUME_FROM:
    ck = torch.load(RESUME_FROM, map_location=DEVICE)
    model.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["optimizer"])
    if ck.get("scheduler"):
        scheduler.load_state_dict(ck["scheduler"])
    start_epoch = ck["epoch"] + 1
    print("resumed from %s at epoch %d" % (RESUME_FROM, start_epoch))

t0 = time.time()
for epoch in range(start_epoch, EPOCHS):
    tr, steps = run_epoch(model, train_loader, optimizer, scaler, train=True)
    if val_loader:
        va, _ = run_epoch(model, val_loader, optimizer, scaler, train=False)
        _, overall, _ = evaluate(model, val_loader)
    else:
        va, overall = float("nan"), {"dice": float("nan"), "iou": float("nan")}
    if steps > 0:                 # optimizer.step() ran this epoch -> correct order to step scheduler
        scheduler.step()
    history.append(dict(epoch=epoch, train_loss=tr, val_loss=va, val_dice=overall["dice"],
                        val_iou=overall["iou"], lr=optimizer.param_groups[0]["lr"],
                        secs=round(time.time() - t0, 1)))
    pd.DataFrame(history).to_csv(CKPT_DIR / ("%s_history.csv" % MODEL), index=False)
    save_ckpt(CKPT_DIR / ("%s_last.pth" % MODEL), model, optimizer, scheduler, epoch, overall)
    if epoch % CKPT_EVERY == 0:
        save_ckpt(CKPT_DIR / ("%s_epoch%03d.pth" % (MODEL, epoch)), model, optimizer, scheduler, epoch, overall)
    if overall["dice"] > best_dice:
        best_dice = overall["dice"]
        save_ckpt(CKPT_DIR / ("%s_best.pth" % MODEL), model, optimizer, scheduler, epoch, overall)
    print("epoch %03d | train %.4f | val %.4f | val_dice %.4f | best %.4f"
          % (epoch, tr, va, overall["dice"], best_dice))
print("done. checkpoints in", CKPT_DIR)

## 6. Save the front-end + sanity check

We reload the **best** epoch, then save the front-end (frozen encoder + trained fusion + 2D→3D lift)
to `models/front_end.pth` — the artifact the decoder pipeline loads and freezes. The neutral head is
discarded. The Dice/IoU below are just a sanity check that the front-end learned something useful;
the real numbers come from the decoder runs.

In [ ]:
best_path = CKPT_DIR / ("%s_best.pth" % MODEL)
if best_path.exists():
    model.load_state_dict(torch.load(best_path, map_location=DEVICE)["model"])
    print("loaded", best_path.name)

# ---- save the trained front-end (frozen encoder + trained fusion + 2D->3D lift) ----
# This is the artifact decoder_pipeline.ipynb loads and freezes wholesale. The neutral head is
# NOT saved -- it was only a scaffold to give the fusion/lift a gradient signal.
torch.save({"front_end": model.fusion.state_dict(),
            "config": {"LIFT_DEPTH": LIFT_DEPTH, "BACKBONE": BACKBONE,
                       "TARGET_RES": TARGET_RES, "GT_THRESH": GT_THRESH}},
           FRONTEND_CKPT)
print("saved front-end ->", FRONTEND_CKPT)

# ---- sanity-check the neutral head (overall + healthy vs fractured) ----
eval_loader = test_loader or val_loader
if eval_loader:
    df, overall, by = evaluate(model, eval_loader)
    print("OVERALL:", {k: round(v, 4) for k, v in overall.items()})
    if by is not None:
        print("\nBY GROUP (healthy vs fractured):\n", by.round(4))
else:
    print("no eval data in this split.")

In [ ]:
h = pd.read_csv(CKPT_DIR / ("%s_history.csv" % MODEL))
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(h.epoch, h.train_loss, label="train"); ax[0].plot(h.epoch, h.val_loss, label="val")
ax[0].set_title("%s loss" % MODEL); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(h.epoch, h.val_dice, label="val Dice"); ax[1].plot(h.epoch, h.val_iou, label="val IoU")
ax[1].set_title("%s val metrics" % MODEL); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout(); plt.show()

## 7. Quality-assurance views

**(A) Tune `GT_THRESH`.** The first row shows the windowed CT slice and the bone mask at a few
thresholds with the resulting occupancy fraction. Pick a threshold that isolates bone (a few % of
voxels) without swallowing soft tissue, set it in the CONFIG cell, and **delete the
`predrr_occupancy_*` cache folder** so it re-binarises.

**(B) Reconstruction preview.** Mid-slices of the prediction next to the GT, as a quick visual
sanity check (it will look rough after a 2-epoch smoke run — that is expected).

In [ ]:
# (A) GT bone-occupancy threshold QA
sample = paired_index.iloc[0]
vol = nib.load(str(gt_file(sample.dataset, sample.case, sample.side))).get_fdata().astype(np.float32)
mid = vol.shape[2] // 2
fig, ax = plt.subplots(1, 4, figsize=(14, 4))
ax[0].imshow(vol[:, :, mid], cmap="gray"); ax[0].set_title("CT (windowed)")
for j, thr in enumerate([0.3, 0.4, 0.5]):
    ax[j + 1].imshow(vol[:, :, mid] > thr, cmap="gray")
    ax[j + 1].set_title("occ>%.1f  (%.1f%%)" % (thr, (vol > thr).mean() * 100))
for a in ax:
    a.axis("off")
plt.suptitle("%s %s %s - current GT_THRESH=%.2f" % (sample.dataset, sample.case, sample.side, GT_THRESH))
plt.tight_layout(); plt.show()

# (B) reconstruction preview vs GT
if eval_loader:
    batch = next(iter(eval_loader))
    with torch.no_grad():
        out, _ = model(batch["ap"].to(DEVICE), batch["lat"].to(DEVICE))
    pred = (torch.sigmoid(out[0, 0]).cpu().numpy() > 0.5).astype(float)
    gtv = batch["gt"][0, 0].numpy()
    m = pred.shape[0] // 2
    fig, ax = plt.subplots(1, 3, figsize=(11, 4))
    ax[0].imshow(batch["ap"][0, 0], cmap="gray"); ax[0].set_title("input AP")
    ax[1].imshow(gtv[m], cmap="gray"); ax[1].set_title("GT mid-slice")
    ax[2].imshow(pred[m], cmap="gray"); ax[2].set_title("prediction mid-slice")
    for a in ax:
        a.axis("off")
    plt.tight_layout(); plt.show()

## Next steps

- This writes `models/front_end.pth`. Next, run `decoder_pipeline.ipynb` (with `FREEZE_FRONTEND=True`)
  for `MODEL="unet"` then `MODEL="vnet"` — both load and freeze this front-end, so the decoder is the
  only difference.
- Tune `GT_THRESH` with the QA cell before the full run, then clear the `predrr_occupancy_*` cache.
- If `256^3` runs out of GPU memory: keep `BATCH_SIZE = 1` and ensure `USE_AMP` and `USE_GRAD_CKPT`
  are `True` (the frozen encoder already cuts activation memory).